# Cardiac Abnormality Assessment from ECG Leads with KaleCardiac

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pykale/cardiac/blob/main/tutorials/mdas_tutorial/tutorial-LSEMVAE.ipynb)

In this tutorial, we use **KaleCardiac** to pretrain, fine-tune, evaluate and interpret **LS-EMVAE** [1], a
lead-specific multimodal variational autoencoder (VAE) that reads a **twelve-lead electrocardiogram (ECG)** as twelve
**modalities**, one per lead. Pretrained on all twelve leads, it is fine-tuned on the **six limb leads** a low-cost
recorder captures -- or on all twelve, with one switch -- to classify each subject as **healthy** or as having a
**cardiothoracic abnormality**.

LS-EMVAE has four core parts, all of them here: **an encoder per lead**; **hierarchical modality expert (HiME)
fusion** of the leads' latent posteriors; **one decoder shared by every lead**; and a **latent alignment** term that
keeps each lead's posterior close to the joint one.

A twelve-lead ECG is usually modelled as one twelve-channel signal, as if its leads were interchangeable channels of
a single view. They are not: each lead projects the heart's electrical activity onto a different axis, so LS-EMVAE
treats them as *different views of the same cardiac state* -- which is what a multimodal VAE is built for. That gives
two things the channel view cannot:

- **an encoder per lead, which keeps working when the other leads are absent** -- what makes a model pretrained on
  twelve leads usable on six, or on one;
- **a principled fusion of the leads' evidence**: a product of experts within groups of leads that see related
  territory, then an average across the groups, so that no one group decides the representation on its own.

The workflow starts from **raw DICOM files**, the format hospitals export ECGs in, and each step is a few calls to the
KaleCardiac API:

- **Step 0: Environment preparation** -- install KaleCardiac, download the tutorial dataset, and load the configuration.
- **Step 1: Multimodal pretraining** -- load the unlabelled twelve-lead ECGs, split each into its twelve leads, and
  pretrain LS-EMVAE on all of them.
- **Step 2: Multimodal fine-tuning** -- load the labelled ECGs' six limb leads, or all twelve, and fine-tune a
  classifier.
- **Step 3: Evaluation** -- score the held-out subjects, with the fine-tuning leads and with fewer.
- **Step 4: Interpretation** -- find which leads, and where in each, the predictions rested on.

**The tutorial dataset** is the CardioVAE tutorial's: 20 subjects, each with a twelve-lead ECG and a chest X-ray
stored as DICOM, in two copies -- `kalecardiac_pretraining`, without labels, and `kalecardiac_finetuning`, with a label
file. This tutorial reads only the ECGs. The dataset exists to show the workflow end to end: the 20 subjects are built
from a handful of recordings, the same subjects serve pretraining and fine-tuning, and the labels are illustrative
rather than clinical findings. **The numbers this notebook prints show that the workflow runs; they do not measure
LS-EMVAE's performance.**

**Estimated runtime:** about 10 minutes on a Colab T4 GPU, installation included.

## Step 0: Environment Preparation

### Google Colab runtime

To run this tutorial in Google Colab, use a GPU runtime: open **Runtime > Change runtime type** and choose
**T4 GPU**. The tutorial also runs on a CPU, more slowly.

### Package installation

The cell below installs KaleCardiac with pip, straight from its GitHub repository, with two optional extras:
`imaging` (pydicom and scikit-image, which PyKale's DICOM reader needs) and `interpret` (Captum, for integrated
gradients). pip brings in everything else the tutorial uses that Colab lacks -- PyKale, PyTorch Lightning -- and the
versions KaleCardiac needs of what Colab has. It then fetches this tutorial's own folder, the notebook and its
`configs`, without the rest of the repository, and works from that folder.

Running locally, skip this cell: install with `pip install -e ".[imaging,interpret]"` from a clone of the
repository, and open the notebook from its folder, `tutorials/mdas_tutorial`.

**Estimated runtime:** 3 minutes

In [ ]:
!pip install --quiet "kalecardiac[imaging,interpret] @ git+https://github.com/pykale/cardiac@main"
!git clone --quiet --depth 1 --filter=blob:none --sparse https://github.com/pykale/cardiac
!git -C cardiac sparse-checkout set tutorials/mdas_tutorial
%cd cardiac/tutorials/mdas_tutorial

### Setup

Warnings and PyTorch Lightning's start-up messages are silenced, so the output shows what matters, and the cell
reports the KaleCardiac it found -- which shows the installation worked.

In [ ]:
import logging
import warnings
from pathlib import Path

import pytorch_lightning

import kalecardiac

warnings.filterwarnings("ignore")
# Importing Lightning sets its loggers' level, so they are quieted after it is imported.
for name in (pytorch_lightning.__name__, "lightning_fabric"):
    logging.getLogger(name).setLevel(logging.ERROR)
print(f"KaleCardiac {kalecardiac.__version__}, working in {Path.cwd()}")

### Dataset download

The tutorial dataset is one ZIP archive on Dropbox, which the CardioVAE tutorial downloads too. The cell below
downloads it into the runtime's local storage -- no Google Drive, and no sign-in -- and unpacks it into `data/`, beside
this notebook; if the CardioVAE tutorial has already unpacked it there, the cell downloads nothing:

```
data/
├── kalecardiac_pretraining/
│   ├── ECG/<subject>/.../*.dcm    one twelve-lead ECG per subject (DICOM ECG waveform)
│   └── CXR/<subject>/.../*.dcm    one chest X-ray per subject, which this tutorial does not read
└── kalecardiac_finetuning/
    ├── ECG/<subject>/.../*.dcm
    ├── CXR/<subject>/.../*.dcm
    └── labels.csv                 subject_id,label
```

Each subject is a folder named by the subject's identifier, holding the subject's DICOM file, as a hospital export
lays them out. The archive holds the data alone: the code and the configurations come with KaleCardiac. To use your
own data instead, skip this cell, and see *Your own data* below.

**Estimated runtime:** 1 minute

In [ ]:
import urllib.request
import zipfile

# The tutorial dataset on Dropbox; "dl=1" asks Dropbox for the file itself rather than its web page.
DATA_URL = "https://www.dropbox.com/scl/fi/y488iouenrk641dxtd8ai/kalecardiac_tutorial_data.zip?rlkey=rorf5mvbt24jdw2cr3ecyy91n&st=yt83r37g&dl=1"

if not Path("data").is_dir():
    archive, _ = urllib.request.urlretrieve(DATA_URL, "kalecardiac_tutorial_data.zip")
    with zipfile.ZipFile(archive) as bundle:
        bundle.extractall("data")
print(sorted(folder.name for folder in Path("data").iterdir()))

### Configuration

KaleCardiac reads every setting from a [YACS](https://github.com/rbgirshick/yacs) configuration, which keeps an
experiment's settings apart from its code. The tutorial's two are in its `configs` folder, each merged onto
KaleCardiac's defaults (`kalecardiac.config`):

- `configs/config_lsemvae_pretrain.yaml` -- the unlabelled dataset, the twelve leads LS-EMVAE reads (the order it
  stacks them in as experts), each lead's length and rate, the lead encoders, the hierarchical fusion, the objective
  with its latent alignment term, and pretraining's schedule.
- `configs/config_lsemvae_finetune.yaml` -- fine-tuning starts from the pretrained model, so it starts from the
  pretraining configuration and adds the labelled dataset, the leads it is read with (the six limb leads;
  `USE_12_LEADS` below switches to all twelve), the endpoint its labels record, the train/validation/test split and
  the fine-tuning schedule.

Change a setting in either file, or override it in the cells below. With the dataset somewhere other than `data/`,
see *Your own data*.

In [ ]:
from kalecardiac.config import get_cfg_defaults

cfg_pt = get_cfg_defaults()
cfg_pt.merge_from_file("configs/config_lsemvae_pretrain.yaml")

# ------ Hyperparameters to play with -----
cfg_pt.MODEL.LATENT_DIM = 128
cfg_pt.FUSION.LATENT_METHOD = "hime"  # "poe" (no average across groups) and "moe" (no product) are its ablations
cfg_pt.OBJECTIVE.ALIGNMENT_WEIGHT = 0.1  # 0 drops the latent alignment term
cfg_pt.SOLVER.MAX_EPOCHS = 10
cfg_pt.SOLVER.BATCH_SIZE = 4
cfg_pt.OBJECTIVE.ANNEALING_EPOCHS = cfg_pt.SOLVER.MAX_EPOCHS  # the KL weight rises over the whole run

print(f"DATASET.ROOT: {cfg_pt.DATASET.ROOT}\n")
for section in ("ECG", "FUSION", "OBJECTIVE", "SOLVER"):
    print(f"{section}:\n{cfg_pt[section]}\n")

In [ ]:
from kalecardiac.loaddata import LIMB_LEADS, STANDARD_12_LEAD

# Fine-tuning starts from the pretrained model, and so from its configuration.
cfg_ft = cfg_pt.clone()
cfg_ft.merge_from_file("configs/config_lsemvae_finetune.yaml")

# ------ Hyperparameters to play with -----
# The task's leads: False fine-tunes on the six limb leads (6L-ECG), as the published model is; True on all twelve.
USE_12_LEADS = False
cfg_ft.ECG.LEADS = list(STANDARD_12_LEAD if USE_12_LEADS else LIMB_LEADS)
cfg_ft.FINETUNE.MAX_EPOCHS = 20
cfg_ft.FINETUNE.BATCH_SIZE = 4
cfg_ft.MODEL.HEAD_HIDDEN = [128]

print(f"DATASET.ROOT: {cfg_ft.DATASET.ROOT}")
print(f"ECG.LEADS: {list(cfg_ft.ECG.LEADS)} -- {len(cfg_ft.ECG.LEADS)} of the {len(cfg_pt.ECG.LEADS)} pretrained")
print(f"ENDPOINT: {cfg_ft.ENDPOINT.NAME}, the {cfg_ft.ENDPOINT.LABEL_COLUMN!r} column of labels.csv\n")
print(f"FINETUNE:\n{cfg_ft.FINETUNE}")

### Your own data

The same steps run on your own ECGs, laid out as the tutorial dataset is: `ECG/`, holding one folder per subject with
that subject's DICOM ECG, and, for fine-tuning, `labels.csv` with a `subject_id` column and a column of classes (0 or
1). Uncomment the cell below and point it at your folders; the dataset download above can then be skipped. Your ECGs'
sampling rate is read from their DICOM headers, and they are resampled to the 500 Hz LS-EMVAE reads. ECGs recorded
with six leads -- a low-cost recorder's -- are fine-tuned on with `USE_12_LEADS = False`.

In [ ]:
# cfg_pt.DATASET.ROOT = "/path/to/your/pretraining_data"  # holds ECG/
# cfg_ft.DATASET.ROOT = "/path/to/your/finetuning_data"  # holds ECG/ and labels.csv
# cfg_ft.ENDPOINT.LABEL_COLUMN = "label"  # the column of labels.csv holding each subject's class

## Step 1: Multimodal Pretraining

LS-EMVAE is pretrained without labels, on the twelve-lead ECGs of `kalecardiac_pretraining`, each lead a modality of
its own.

### Data loading

The ECGs are a folder holding one folder per subject, with the subject's DICOM file beneath it.
`ECGFileSource.from_dicom`, in `kalecardiac.loaddata`, indexes that folder by subject and reads a subject's file when
it is needed, so a cohort of any size loads the same way. Each file is read with PyKale's [3] DICOM reader,
[`kale.loaddata.image_access.read_dicom_phases`](https://github.com/pykale/pykale/blob/main/kale/loaddata/image_access.py), and the twelve-lead rhythm waveform is taken out of
it in millivolts, with each lead's name and the sampling rate read from the file's header. Every other subject's
leads are found by *name*, so a file that stores aVF before aVL cannot transpose them -- which matters all the more
when each lead has an encoder of its own.

**Estimated runtime:** 10 seconds

In [ ]:
from kalecardiac.loaddata import ECGFileSource

ecg_dir_pt = Path(cfg_pt.DATASET.ROOT, "ECG")
ecg_raw = ECGFileSource.from_dicom(ecg_dir_pt)
print(ecg_raw)

One subject's ECG, as loaded: its twelve leads, which LS-EMVAE reads as twelve modalities -- the limb leads on the
left, the precordial leads on the right.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

subject = sorted(ecg_raw.paths)[0]
recording = ecg_raw.get(subject, 0).numpy()
time = np.arange(recording.shape[1]) / ecg_raw.sampling_rate

fig, axes = plt.subplots(6, 2, figsize=(15, 9), sharex=True)
for index, lead in enumerate(cfg_pt.ECG.LEADS):
    ax = axes[index % 6, index // 6]
    ax.plot(time, recording[ecg_raw.leads.index(lead)], color="black", linewidth=0.6)
    ax.set_ylabel(lead, rotation=0, labelpad=15, fontsize="large")
for ax in axes[-1]:
    ax.set_xlabel("Time (seconds)")
fig.suptitle(f"{subject}: the twelve leads, as recorded (mV)", fontsize="x-large")
plt.tight_layout()
plt.show()

### Data preparation

LS-EMVAE reads every lead as ten seconds at 500 Hz -- 5,000 samples -- normalized lead by lead. Two KaleCardiac calls
build that from the configuration:

- `get_ecg_transform_from_cfg` (`kalecardiac.auto`) reads `cfg.ECG`: fill any dropped samples, resample if the
  recording arrives at another rate, crop or pad each lead to ten seconds, and normalize each lead to zero mean and
  unit variance. Every step works on one lead at a time, so none mixes one lead into another, and with
  `ECG.NUM_CHANNELS` 0 the leads stay apart, a channel each, rather than interleaved into one strip as CardioVAE reads
  them. The leads are taken by name in `ECG.LEADS`' order, whichever order a file stores them in.
- `get_lead_sources` (`kalecardiac.loaddata`) splits the recording into one source per lead, named by the lead.

`MultimodalDataset` then treats the twelve leads as it treats any set of modalities: each subject yields twelve, `I`
to `V6`, each a single lead of 5,000 samples. Nothing after this point knows that the modalities happen to be leads.

**Estimated runtime:** 10 seconds

In [ ]:
from kalecardiac.auto import get_ecg_transform_from_cfg
from kalecardiac.loaddata import MultimodalDataset, get_lead_sources

# The rate the recordings were made at, as their DICOM headers state it, so any other is resampled.
cfg_pt.ECG.SOURCE_SAMPLING_RATE = ecg_raw.sampling_rate
ecg_transform = get_ecg_transform_from_cfg(cfg_pt)

ecg = ECGFileSource.from_dicom(ecg_dir_pt, select=cfg_pt.ECG.LEADS, transform=ecg_transform)
pretrain_set = MultimodalDataset(sorted(ecg.paths), get_lead_sources(ecg, leads=cfg_pt.ECG.LEADS))

print(pretrain_set)
print({name: tuple(value.shape) for name, value in pretrain_set[0].modalities.items()})

### Multimodal pretraining

`LSEMVAE`, in `kalecardiac.model.embed`, has one variational encoder per lead -- a 1-D convolutional network, each
giving a Gaussian over the latent space all the leads share -- and **one decoder shared by every lead**, which forces
that latent to carry what the leads have in common rather than what each could keep to itself. Its fusion is
**hierarchical modality experts (HiME)**: the leads' Gaussians and a standard-normal prior are multiplied within
groups of leads that see related territory -- the prior with I, II and III; aVR, aVL and aVF; V1 to V3; V4 to V6 --
and the four groups are then averaged. A subject missing a lead contributes no expert for it. Its input length comes
from the prepared data.

`pretrain`, in `kalecardiac.auto`, trains it with the configured objective -- the ELBO over every lead, with the KL
term annealed in, and a **latent alignment** term that pulls each lead's posterior towards the joint one, so that a
lead's representation stays usable when the other leads are absent -- and saves the weights to `pretrained.pt` in
`OUTPUT.OUT_DIR`. It needs no labels: pretraining is a reconstruction task.

**Estimated runtime:** 1 to 2 minutes

In [ ]:
import pandas as pd

from kalecardiac.auto import pretrain
from kalecardiac.model.embed import LSEMVAE
from kalecardiac.utils import set_seed

set_seed(cfg_pt.SOLVER.SEED)
model = LSEMVAE(
    leads=cfg_pt.ECG.LEADS,
    length=ecg.length,
    latent_dim=cfg_pt.MODEL.LATENT_DIM,
    channels=cfg_pt.MODEL.CHANNELS,
    kernel_size=cfg_pt.MODEL.KERNEL_SIZE,
    fusion=cfg_pt.FUSION.LATENT_METHOD,
    num_groups=cfg_pt.FUSION.NUM_GROUPS,
    use_prior=cfg_pt.FUSION.USE_PRIOR,
    shared_decoder=True,  # False gives each lead a decoder of its own: the baseline LS-EMVAE is compared with
)
print(model)
print(f"{sum(parameter.numel() for parameter in model.parameters()):,} parameters")

pretrained = pretrain(cfg_pt, model, pretrain_set, cfg_pt.OUTPUT.OUT_DIR)

# The training loss per epoch, from the run's log.
log = max(Path(cfg_pt.OUTPUT.OUT_DIR).glob("pretrain/history/*/metrics.csv"), key=lambda path: path.stat().st_mtime)
pd.read_csv(log).groupby("epoch")["train_loss"].mean().round(1).to_frame("training loss").T

Before any label is used, one check shows whether pretraining did what fine-tuning on fewer leads relies on.
`collect_latents`, in `kalecardiac.interpret`, encodes every subject and returns the mean of the joint posterior and
of each lead's own. Where the alignment term did its job, each lead's mean sits close to the joint one -- one lead
alone points to much the same place as all twelve together -- and a lead far from it is one whose encoder has gone its
own way.

In [ ]:
from kalecardiac.auto import get_dataloader
from kalecardiac.interpret import collect_latents

loader = get_dataloader(cfg_pt, pretrain_set, cfg_pt.SOLVER.BATCH_SIZE)
_, joint, per_lead = collect_latents(pretrained, loader, per_modality=True)
distance = pd.Series({lead: np.mean((mean - joint) ** 2) for lead, mean in per_lead.items()})

ax = distance.plot.bar(figsize=(10, 3), color="steelblue", rot=0)
ax.set(ylabel="Mean squared distance", title="How far each lead's posterior mean sits from the joint one")
plt.tight_layout()
plt.show()

## Step 2: Multimodal Fine-tuning

The pretrained lead encoders are fine-tuned on the labelled ECGs of `kalecardiac_finetuning` to classify each subject:
**0** healthy, **1** cardiothoracic abnormality. By default they read the **six limb leads** alone, as a low-cost
recorder would capture them -- the 6L-ECG setting LS-EMVAE was built for: pretrained on twelve leads, fine-tuned on
six, each of the six keeping the encoder it learned alongside the others. `USE_12_LEADS = True`, in the configuration
above, fine-tunes on all twelve instead (12L-ECG).

### Data loading

The labelled cohort has the pretraining layout plus `labels.csv`, one row per subject. Its ECGs load through the same
DICOM source, with the same transform -- the pretrained encoders read exactly that shape -- taking only the leads
`cfg_ft.ECG.LEADS` names, and the labels are an ordinary table.

**Estimated runtime:** 1 second

In [ ]:
ecg_ft = ECGFileSource.from_dicom(Path(cfg_ft.DATASET.ROOT, "ECG"), select=cfg_ft.ECG.LEADS, transform=ecg_transform)
labels = pd.read_csv(Path(cfg_ft.DATASET.ROOT, "labels.csv"))

print(ecg_ft)
labels.head()

### Data preparation

`get_lead_sources` splits the fine-tuning leads into a modality each, as in pretraining, and three calls turn the
labels into a fine-tuning run, each reading `cfg_ft`:

- `derive_endpoint` reads each subject's class from `ENDPOINT.LABEL_COLUMN`, dropping a subject with none rather than
  assuming it negative, and `get_endpoint_target` makes the class the supervision every batch carries.
- `get_folds` divides the subjects into training, validation and test sets, stratified by label, with no subject in
  two of them: 20% are held out for testing, and the validation set that chooses the checkpoint is carved out of the
  rest -- never out of the test set.

The class weight the loss uses is taken from the training set alone, when the fine-tuning task is built.

**Estimated runtime:** 1 second

In [ ]:
from kalecardiac.auto import derive_endpoint, get_endpoint_target, get_folds

finetune_leads = get_lead_sources(ecg_ft, leads=cfg_ft.ECG.LEADS)
table = derive_endpoint(cfg_ft, labels)
target = get_endpoint_target(cfg_ft, table, id_column="subject_id")
fold = get_folds(cfg_ft, table)[0]
print(f"modalities: {list(finetune_leads)}")
print(
    f"{len(fold.train_ids)} subjects to train on, {len(fold.val_ids)} to validate on, {len(fold.test_ids)} to test on"
)

### Multimodal fine-tuning

`fine_tune`, in `kalecardiac.auto`, fine-tunes as the published LS-EMVAE is: it keeps the pretrained encoder of each
lead it is given, frozen, as a feature extractor, and trains a classification head -- one hidden layer of 128 units,
with dropout -- on the leads' posterior means, concatenated. The encoders of leads it is not given are not used.
HiME, the shared decoder and the alignment term did their work in pretraining: they shaped every lead's encoder, so
that its mean already sits in the coordinates all the leads share, which is what lets six of the twelve encoders
carry the task alone. The head, the loss and the positive class's weight follow from the endpoint and the training
labels. After training, the checkpoint that scored best on the validation subjects is restored, and the test
subjects are scored.

**Estimated runtime:** 1 minute

In [ ]:
from kalecardiac.auto import fine_tune

set_seed(cfg_ft.SOLVER.SEED)
result = fine_tune(cfg_ft, pretrained, finetune_leads, target, fold, cfg_ft.OUTPUT.OUT_DIR)

## Step 3: Evaluation

`fine_tune` has scored the held-out test subjects with `score_predictions`, which chooses the metrics an endpoint
calls for -- for a binary one, `binary_metrics` in `kalecardiac.evaluate`:

- **Accuracy:** the share of subjects classified correctly.
- **AUROC:** the area under the receiver operating characteristic curve -- how well the scores rank positive subjects
  above negative ones, at every threshold at once.
- **MCC:** the Matthews correlation coefficient, which stays informative when the classes are imbalanced.
- **Sensitivity** and **specificity:** the shares of positive and of negative subjects classified correctly.

The predictions are kept per subject, so they can be inspected, or scored again at another threshold. With only a
few test subjects, the scores can rank them correctly -- a high AUROC -- and still all fall on one side of the 0.5
threshold that accuracy, MCC, sensitivity and specificity cut at. On this dataset -- 20 subjects built from a handful
of recordings -- the numbers show that the evaluation runs, not how well LS-EMVAE performs.

In [ ]:
metrics = pd.Series(result.metrics)[["accuracy", "roc_auc", "mcc", "sensitivity", "specificity"]].astype(float)
metrics.rename({"roc_auc": "AUROC", "mcc": "MCC"}).round(3).to_frame("test")

In [ ]:
predictions = pd.DataFrame(result.predictions.rows())
predictions.assign(probability=1 / (1 + np.exp(-predictions["score"]))).round(3)

**With fewer leads.** A recorder may capture fewer leads than the model was fine-tuned on: a single-lead wearable
records lead I alone. `progressive_ablation`, in `kalecardiac.interpret`, withholds the leads one after another and
scores the test subjects again at each step. A lead is withheld through the presence mask, so the model sees an
absent lead -- exactly what a recorder that never captured it produces -- rather than a flat line, and each lead's
encoder works without the others.

In [ ]:
from kalecardiac.auto import get_ablation_metric
from kalecardiac.interpret import progressive_ablation

# Withheld from the last lead backwards, leaving the first: lead I, which a single-lead wearable records.
WITHHOLD = list(reversed(cfg_ft.ECG.LEADS[1:]))

_, metric, key = get_ablation_metric(cfg_ft)
steps = progressive_ablation(result.model, result.test_loader, metric, key, order=WITHHOLD)
pd.DataFrame([{"leads": " ".join(step["present"]), "test AUROC": step["score"]} for step in steps]).round(3)

## Step 4: Interpretation

Two questions, both answered from the fine-tuned model:

- **Which leads did it rely on?** `interpret_model`, in `kalecardiac.auto`, runs integrated gradients over the test
  subjects and reports each lead's *attribution ratio* -- the share of its samples that were strongly attributed --
  and that ratio's share of the total. It then withholds each lead in turn, through the presence mask, and scores the
  test set again: a lead whose loss costs nothing is one the others could stand in for. The two can disagree, and the
  disagreement is informative: a lead can be heavily attributed and still be redundant, because another lead carries
  the same information -- as leads, all projections of one heart's activity, often do.
- **Where in each lead?** `modality_attributions`, in `kalecardiac.interpret`, attributes one subject's prediction to
  every sample of each of its leads with [Captum](https://captum.ai)'s integrated gradients [2], and
  `normalize_attribution` rescales the leads' maps together to [0, 1], so the leads can be compared and the regions
  above a threshold drawn.

**Estimated runtime:** 10 seconds

In [ ]:
from kalecardiac.auto import interpret_model

summary = interpret_model(cfg_ft, result.model, result.test_loader, cfg_ft.OUTPUT.OUT_DIR)
ablation = summary.get("ablation", {})
lead_importance = pd.DataFrame(summary["attribution"]).T
lead_importance["test AUROC without it"] = [ablation.get(f"without_{lead}") for lead in lead_importance.index]
print(f"Test AUROC with all {len(lead_importance)} leads: {ablation.get('full', float('nan')):.3f}")
lead_importance.round(3)

One test subject's prediction, attributed to each of its leads.

In [ ]:
import torch

from kalecardiac.interpret import modality_attributions, normalize_attribution
from kalecardiac.loaddata import collate_subjects

# Change these to explain another test subject or lead, to move the threshold, or to widen the zoom.
SAMPLE_IDX, LEAD = 0, "II"
THRESHOLD, ZOOM_SECONDS = 0.75, 0.5

sample = collate_subjects([result.test_loader.dataset[SAMPLE_IDX]])
attributions = modality_attributions(result.model.model, sample.modalities, present=sample.present)

# One row per lead. The maps are normalized together, so one lead's importance can be read against another's.
leads = list(cfg_ft.ECG.LEADS)
ecg_signals = np.stack([sample.modalities[lead][0, 0].numpy() for lead in leads])
ecg_importance = normalize_attribution(np.stack([attributions[lead][0, 0] for lead in leads]))
ecg_time = np.arange(ecg_signals.shape[1]) / cfg_ft.ECG.SAMPLING_RATE

probability = torch.sigmoid(result.model.predict(sample).prediction).item()
print(
    f"{sample.subject_id[0]}: label {int(sample.target['label'])}, predicted probability of a cardiothoracic "
    f"abnormality {probability:.2f}"
)

**Full ECG view:** each lead, with the samples whose attribution clears the threshold drawn in red.

In [ ]:
from matplotlib.lines import Line2D


def plot_leads(names: list[str], window: slice, title: str, width: float = 1.0):
    fig, axes = plt.subplots(len(names), 1, figsize=(12, 1.5 * len(names) + 1.5), sharex=True, squeeze=False)
    for ax, name in zip(axes[:, 0], names, strict=True):
        signal, importance = ecg_signals[leads.index(name), window], ecg_importance[leads.index(name), window]
        ax.plot(ecg_time[window], signal, color="black", linewidth=width)
        for index in np.flatnonzero(importance >= THRESHOLD):
            stretch = slice(max(index - 2, 0), index + 3)
            ax.plot(ecg_time[window][stretch], signal[stretch], color="red", linewidth=2 * width)
        ax.set_ylabel(name, rotation=0, labelpad=20, fontsize="large")
    axes[-1, 0].set_xlabel("Time (seconds)", fontsize="x-large")
    legend = [Line2D([], [], color="black"), Line2D([], [], color="red", linewidth=2)]
    axes[0, 0].legend(legend, ["ECG waveform", "Important regions"], fontsize="medium", loc="best")
    axes[0, 0].set_title(title, fontsize="x-large")
    plt.tight_layout()
    plt.show()


plot_leads(leads, slice(None), "Full ECG: each lead, with its important regions")

**Zoomed-in ECG segment:** half a second of one lead -- lead II by default -- around its most strongly attributed
sample, in finer detail.

In [ ]:
centre = ecg_time[np.argmax(ecg_importance[leads.index(LEAD)])]
first = max(centre - ZOOM_SECONDS / 2, 0.0)
last = first + ZOOM_SECONDS
window = slice(int(first * cfg_ft.ECG.SAMPLING_RATE), int(last * cfg_ft.ECG.SAMPLING_RATE))
plot_leads([LEAD], window, f"Zoomed-in ECG, lead {LEAD} ({first:.2f} s - {last:.2f} s)", width=3)

## Where next

* The [LS-EMVAE example](https://github.com/pykale/cardiac/tree/main/examples/lsemvae_ecg) runs this model from the
  command line, with cross-validation, a comparison of lead configurations, the published ablations as settings, the
  robustness curve and a written report: `python -m examples.lsemvae_ecg.main --cfg <your config>`.
* The [CardioVAE tutorial](https://github.com/pykale/cardiac/blob/main/tutorials/mdas_tutorial/tutorial-CardioVAE.ipynb)
  fuses the ECG with a chest X-ray instead, on the same dataset.
* The [API overview](https://github.com/pykale/cardiac/blob/main/API_overview.md) lists every stage of the
  KaleCardiac API.

## References

[1] Suvon, M. N. I., et al. (2025). Multimodal latent fusion of ECG leads for early assessment of pulmonary
hypertension. *arXiv preprint arXiv:2503.13470*. Reference implementation: https://github.com/Shef-AIRE/LS-EMVAE.

[2] Sundararajan, M., Taly, A., & Yan, Q. (2017). Axiomatic attribution for deep networks. In *International
Conference on Machine Learning* (pp. 3319-3328).

[3] Lu, H., Liu, X., Zhou, S., Turner, R., Bai, P., Koot, R. E., ... & Xu, H. (2022). PyKale: Knowledge-aware machine
learning from multiple sources in Python. In *Proceedings of the 31st ACM International Conference on Information &
Knowledge Management* (pp. 4274-4278).